* streamlit for web interface
* streamlit-image-coordinates for extracting coordinates from clicks
* ultralytics for sam3
* github realtime-detection-yolo26 for yoloe-26l-seg
* opencv-python to process images in python
* shutil: [documentation](https://docs.python.org/3/library/shutil.html)
* glob: [documentation](https://docs.python.org/3/library/glob.html)
* subprocess to use the models (grounding dino, sam2) in [RFI anomaly detection](https://github.com/Aleinno001/RFI_anomaly_detection/tree/master) without weighing down the web application too much, together with yolo26 and sam3

In [ ]:
#lybraries
!pip install -q streamlit streamlit-image-coordinates ultralytics opencv-python

#YOLO26
!mkdir -p /content/realtime-detection-yolo26
!wget -nc -q https://github.com/ultralytics/assets/releases/download/v8.4.0/yoloe-26l-seg.pt -P /content/realtime-detection-yolo26/

#SAM3
!wget -nc https://huggingface.co/bodhicitta/sam3/resolve/main/sam3.pt -P /content/

#anomaly detection
!git clone https://github.com/Aleinno001/RFI_anomaly_detection.git
!cp /content/drive/MyDrive/thesis_models/main.py /content/RFI_anomaly_detection/anomaly_detection.py


app.py

In [ ]:
%%writefile app.py
import streamlit as st
from streamlit_image_coordinates import streamlit_image_coordinates
import cv2
import numpy as np
import tempfile
from PIL import Image
from ultralytics import YOLO, SAM
import os
import glob
import shutil
import subprocess

def extract_anomalous_frames(file_path):
    if not os.path.exists(file_path):
        return []
    with open(file_path, 'r') as file:
        rows = file.readlines()
        frames_list = [int(row.strip()) for row in rows if row.strip().isdigit()]
        return frames_list

st.set_page_config(layout="wide", page_title="Thermal Anomalies Annotation")

st.title("Anomaly Annotation System for Thermal Videos")

#session_state inisialisation
#save clicks
if 'positive_seeds' not in st.session_state:
    st.session_state.positive_seeds = {}
if 'negative_seeds' not in st.session_state:
    st.session_state.negative_seeds = {}
if 'last_click' not in st.session_state:
    st.session_state.last_click = None

#counting frames for slider
if 'total_frames' not in st.session_state:
  st.session_state.total_frames = 0

#save pre-processed frames
if 'suspect_frames' not in st.session_state:
    st.session_state.suspect_frames = []
if 'current_frame' not in st.session_state:
    st.session_state.current_frame = 0

#callback for changes on uploaded file
def reset_state():
    st.session_state.positive_seeds = {}
    st.session_state.negative_seeds = {}
    st.session_state.last_click = None
    st.session_state.output_video = None
    st.session_state.total_frames = 0
    st.session_state.suspect_frames = []
    st.session_state.current_frame = 0

#initilise processed video as empty
if 'output_video' not in st.session_state:
  st.session_state.output_video = None
if 'output_zip' not in st.session_state:
    st.session_state.output_zip = None

#sidebar menu
with st.sidebar:
    st.header("MENU")
    uploaded_video = st.file_uploader("Upload thermal video (.mp4)",
        type=['mp4'],
        on_change = reset_state)

    #reset annotations
    if st.button("Reset Annotations"):
        st.session_state.positive_seeds = {}
        st.session_state.negative_seeds = {}
        st.session_state.last_click = None
        st.rerun()

    #download video
    if st.session_state.output_video is not None:
      clean_name = os.path.splitext(uploaded_video.name)[0]
      st.download_button(
          label="Download annotated video (.mp4)",
          data=st.session_state.output_video,
          file_name=clean_name+"_annotated.mp4",
          mime="video/mp4",
          on_click="ignore",
          icon=":material/download:",
          icon_position="right",
          width="stretch"
      )
    else:
      st.download_button(
              label="Download annotated video (.mp4)",
              data="dummy",
              icon=":material/download:",
              icon_position="right",
              disabled=True,
              width="stretch"
          )

    #donwload zip
    if st.session_state.output_zip is not None:
      clean_name = os.path.splitext(uploaded_video.name)[0]
      st.download_button(
            label="Download annotations (.zip)",
            data=st.session_state.output_zip,
            file_name=clean_name+"_labels.zip",
            mime="application/zip",
            on_click="ignore",
            icon=":material/download:",
            icon_position="right",
            width="stretch"
        )
    else:
        st.download_button(
                label="Download annotations (.zip)",
                data="dummy",
                icon=":material/download:",
                icon_position="right",
                disabled=True,
                width="stretch"
        )

#extracting frame when video is uploaded
frame_rgb = None
if uploaded_video is not None:
    uploaded_video.seek(0) #rewind to initial frame

    #creating temporary file
    tfile = tempfile.NamedTemporaryFile(delete=False, suffix='.mp4')
    tfile.write(uploaded_video.read())
    tfile.close()

    #debugging: showing size of read video
    st.sidebar.write(f"Video dimensions: {os.path.getsize(tfile.name)} bytes")

    cap = cv2.VideoCapture(tfile.name) #opening the video file
    if st.session_state.total_frames == 0:
      st.session_state.total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) #save total frames
    cap.set(cv2.CAP_PROP_POS_FRAMES, st.session_state.current_frame) #select frame
    ret, frame = cap.read() #(return) ret=True: frame read succesfully
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) #converting bgr to rgb
    cap.release()

col_img, col_ctrl = st.columns([7, 3]) #70% image, 30% controls

#controls: right column
with col_ctrl:
    st.header("Settings")

    #choosing the model
    model_choice = st.selectbox(
        "Select the model:",
        ["YOLOE-26L-Seg", "SAM 3"]
    )

    yolo_prompt = ""
    if model_choice == "YOLOE-26L-Seg":
        yolo_prompt = st.text_input("Enter objects to detect (comma separated):", value="person, animal")
        #st.caption("Example: person, dog, bright object, hot silhouette")

    st.divider()

    #anomaly detection
    st.subheader("Pre-processing (AD)")
    if st.button(label="Start RFI Anomaly Detection", width="stretch"):
      if uploaded_video is None:
        st.error("Upload video (MENU) o start detection.")
      else:
        with st.spinner(text="RFI anomaly detection in progress..."):
          ad_results = subprocess.run(["python", "RFI_anomaly_detection/anomaly_detection.py", "--input_video", tfile.name],
                                      capture_output=True, text=True)

          #debugging
          st.text_area("RFI AD's logs (Grounding DINO):", ad_results.stdout, height=300) #logs

          if ad_results.stderr: #errors
            st.error("RFI Anomaly Detection has generated an error:")
            st.code(ad_results.stderr)

          st.session_state.suspect_frames = extract_anomalous_frames("found_anomalies.txt")

          if len(st.session_state.suspect_frames) > 0:
            st.session_state.current_frame = st.session_state.suspect_frames[0]
            st.success("Anomalies found!")
          else:
            st.info("No anomalies detected.")
          st.rerun()

    st.divider()

    #HITL
    st.subheader("Navigation and Cursor")

    #frame navigator among suspect frames (from AD)
    if len(st.session_state.suspect_frames) > 0:
        selector = st.selectbox(
            label="Select suspect frame to view:",
            options=st.session_state.suspect_frames,
            index=st.session_state.suspect_frames.index(st.session_state.current_frame) if st.session_state.current_frame in st.session_state.suspect_frames else 0
        )
        if selector != st.session_state.current_frame:
            st.session_state.current_frame = selector
            st.rerun()
    #manual frame navigator
    if st.session_state.total_frames > 0:
      manual_frame = st.slider(
          label="Scroll video manually",
          min_value=0,
          max_value=st.session_state.total_frames-1,
          value=st.session_state.current_frame,
      )
      if manual_frame != st.session_state.current_frame:
            st.session_state.current_frame = manual_frame
            st.rerun()
    else:
        st.write("Showing frame 0.")

    #choosing seed (+ive or -ive)
    seed_type = st.radio(
        label="Seed:",
        options=["Positive", "Negative"]
    )

    st.divider()

    #visualising clicked coordinates
    st.write("**Coordinates:**")
    st.write(f"Positives: {st.session_state.positive_seeds}")
    st.write(f"Negatives: {st.session_state.negative_seeds}")

    st.divider()

    #execution
    st.subheader("Processing")
    if st.button("Process video", width="stretch", type="primary"):
        if uploaded_video is None:
          st.error("Upload a video from the MENU first.")
        elif model_choice == "SAM 3" and len(st.session_state.positive_seeds) == 0:
            st.error("SAM 3 requires at least one Positive Seed to start.")
        else:
          with st.spinner(text=f"Video is beeing processed by {model_choice}..."):
            video_to_analyse = tfile.name #temp file for the video uploaded by the user

            #setup for progress bar and video extraction
            cap = cv2.VideoCapture(video_to_analyse)

            progress_text = st.empty()
            progress_bar = st.progress(0)

            if model_choice == "YOLOE-26L-Seg":
              cap.release()
              path_yolo = "/content/realtime-detection-yolo26/yoloe-26l-seg.pt"
              model = YOLO(path_yolo)

              #open vocabulary: defining classes to find
              custom_classes = [c.strip() for c in yolo_prompt.split(",") if c.strip()]
              if custom_classes:
                  model.set_classes(custom_classes)

              results = model.predict(
                source=video_to_analyse,
                conf=0.15, #lowered confidence to allow it to show masks
                save=True, #save video
                save_txt=True, #save annotations in standard YOLO format (.txt)
                project="/content/results",
                name="yolo_out",
                exist_ok=True,
                stream=True #for progress bar
              )

              #updating progress bar
              processed_frames = 0
              for frame_result in results:
                processed_frames += 1
                if processed_frames % 10 == 0 or processed_frames == st.session_state.total_frames:
                  progress_bar.progress(min(processed_frames / st.session_state.total_frames, 1.0))
                  progress_text.write(f"YOLO is scanning frame {processed_frames} of {st.session_state.total_frames}...")

            elif model_choice == "SAM 3":
              model = SAM("sam3.pt")

              #delete old SAM masks
              sam_out_path = "/content/results/sam_out"
              if os.path.exists(sam_out_path):
                shutil.rmtree(sam_out_path)

              #temporary folder to extract clean frames
              temp_frames_dir = "/content/temp_sam_frames"
              if os.path.exists(temp_frames_dir):
                shutil.rmtree(temp_frames_dir)
              os.makedirs(temp_frames_dir, exist_ok=True)

              #find clicked frames
              frames_to_process = [f for f, seeds in st.session_state.positive_seeds.items() if len(seeds) > 0]
              processed_count = 0

              #extract frames from video
              for f_idx in frames_to_process:
                cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
                ret, frame = cap.read()
                if not ret: continue

                #save frame as .jpg image
                img_path = os.path.join(temp_frames_dir, f"frame_{f_idx}.jpg")
                cv2.imwrite(img_path, frame)

                #exatract clicks on f_idx frame
                clicks = [[p[0], p[1]] for p in st.session_state.positive_seeds[f_idx]]
                seed_labels = [1] * len(clicks) #1 = positive

                results = model.predict(
                  source=img_path,
                  points=clicks,
                  labels=seed_labels,
                  save=True, #save video
                  save_txt=True, #save masks in standard YOLO format (.txt)
                  project="/content/results",
                  name="sam_out",
                  exist_ok=True,
                )

                #updating progress bar
                processed_count +=1
                progress_bar.progress(processed_count / len(frames_to_process))
                progress_text.write(f"SAM 3 is annotating frame {f_idx}")

              cap.release()

            progress_text.success("Processing complete.")

            output_folder = "/content/results/yolo_out" if model_choice == "YOLOE-26L-Seg" else "/content/results/sam_out"

            #file video output (YOLO)
            if model_choice == "YOLOE-26L-Seg":
              generated_video_files = [f for f in glob.glob(f"{output_folder}/*") if f.endswith('.mp4') or f.endswith('.avi')]
              if generated_video_files:
                recent = max(generated_video_files, key=os.path.getctime)
                with open(recent, "rb") as video_file:
                  st.session_state.output_video = video_file.read()

            else:
              st.session_state.output_video = None

            #zip dataset containing images and labels (YOLO and SAM)
            if os.path.exists(output_folder):
              path_zip = "/content/results/annotations_dataset"
              shutil.make_archive(path_zip, 'zip', output_folder)
              with open(path_zip + ".zip", "rb") as zip_file:
                st.session_state.output_zip = zip_file.read()

          st.rerun()

#image: left column
with col_img:
  cf = st.session_state.current_frame
  st.header(f"Frame {cf}")

  if frame_rgb is not None:
    #image with planted seed saved
    img_to_draw = frame_rgb.copy()

    #create seeds list for current frame
    if cf not in st.session_state.positive_seeds:
      st.session_state.positive_seeds[cf] = []
    if cf not in st.session_state.negative_seeds:
      st.session_state.negative_seeds[cf] = []

    for p in st.session_state.positive_seeds[cf]:
        cv2.circle(img_to_draw, (p[0], p[1]), 5, (0, 255, 0), -1) #green
    for n in st.session_state.negative_seeds[cf]:
        cv2.circle(img_to_draw, (n[0], n[1]), 5, (255, 0, 0), -1) #red

    value = streamlit_image_coordinates(
        Image.fromarray(img_to_draw),
        key=f"pil_{cf}"
    )

    #saving the coordinates in current frame
    if value is not None and value != st.session_state.last_click: #user has clicked
        st.session_state.last_click = value
        point = (value["x"], value["y"])

        if "Positive" in seed_type:
          if point not in st.session_state.positive_seeds[cf]:
            st.session_state.positive_seeds[cf].append(point)
            st.rerun() #update the interface to show drawn points
        else:
          if point not in st.session_state.negative_seeds[cf]:
            st.session_state.negative_seeds[cf].append(point)
            st.rerun()
  else:
    st.info("Upload a video (MENU) to start annotating.")

ngrok



In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok

#closing old tunnels
ngrok.kill()

#insert token
ngrok.set_auth_token("3BCpYV0SGpgBl8lgMTYc6IAhJof_6aeUhfdwsNSg7iLeHsmnv")

#run streamlit app
get_ipython().system_raw('streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false &')

#open tunnel
public_url = ngrok.connect(8501).public_url

print(f"your url is: {public_url}")

Cloudeflare

In [ ]:
import time

#close old tunnels
!pkill -f streamlit
!pkill -f cloudflared

#run streamlit app
get_ipython().system_raw('streamlit run app.py --server.address 0.0.0.0 &')

#download Cloudflare
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

#open tunnel
get_ipython().system_raw('./cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1 &')

time.sleep(8)

#get link
print("\n LINK BELOW")
!grep -o 'https://[-0-9a-z]*\.trycloudflare\.com' tunnel.log